# 03 - Train MLP from Pre-extracted Concerto Features

This notebook trains the translation head from the `.npz` features saved by `notebooks/02_feature_extraction.ipynb`.

It follows the same Colab logic as notebook 02:
- mount Drive and sync the repo branch
- symlink the shared Drive folders
- install and run the training stack with `python3.10`
- verify that the extracted features match the training config
- generate CLIP label embeddings if missing
- train the MLP and inspect checkpoints/history


### 1. Mount Drive and Sync the Repo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'dev/enc-mlp'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline


### 2. Symlink Shared Drive Folders


In [ ]:
!rm -f data checkpoints features
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features


### 3. Install the Python 3.10 Training Stack
Unlike notebook 02, training does not need `Concerto`, `spconv`, or `torch-scatter`.


In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3.10 python3.10-dev python3.10-distutils
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10
!python3.10 --version
!python3.10 -m pip install --no-cache-dir --force-reinstall torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124
!python3.10 -m pip install --no-cache-dir numpy==1.26.4 pyyaml open_clip_torch python-dotenv huggingface_hub scipy
!python3.10 -c "import torch, yaml, open_clip, dotenv; print('torch', torch.__version__, torch.version.cuda); print('yaml ok'); print('open_clip ok'); print('dotenv ok')"


### 4. Verify Training Inputs
This checks that the extracted features exist and that their feature dimension matches the training config.


In [ ]:
%%bash
set -e
cd /content/Deep_learning_project
PYTHONPATH=/content/Deep_learning_project python3.10 - <<'PY'
from pathlib import Path
import numpy as np
import yaml

features_dir = Path('features/s3dis_area5')
clip_emb_path = Path('data/s3dis_processed/label_to_clip_embeddings.npy')
ckpt_dir = Path('checkpoints/mlp_s3dis')
config_path = Path('configs/train_mlp_s3dis.yaml')
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
expected_dim = int(cfg['model']['input_dim'])
feature_files = sorted(features_dir.glob('*.npz'))

print('features dir exists  :', features_dir.exists(), features_dir)
print('num feature files    :', len(feature_files))
print('sample files         :', [path.name for path in feature_files[:5]])
print('CLIP embeddings file :', clip_emb_path.exists(), clip_emb_path)
print('checkpoint dir exists:', ckpt_dir.exists(), ckpt_dir)
print('expected input dim   :', expected_dim)

if not feature_files:
    raise RuntimeError('No .npz feature files found in features/s3dis_area5. Run notebook 02 first.')

with np.load(feature_files[0]) as sample:
    keys = list(sample.files)
    print('\nsample keys:', keys)
    for key in keys:
        print(key, sample[key].shape, sample[key].dtype)
    feat_key = next((key for key in ('features', 'feature', 'feat') if key in keys), None)
    label_key = next((key for key in ('labels', 'label', 'segment') if key in keys), None)
    if feat_key is None:
        raise RuntimeError(f'Missing feature array in {feature_files[0].name}.')
    if label_key is None:
        raise RuntimeError(f'Missing label array in {feature_files[0].name}.')
    actual_dim = int(sample[feat_key].shape[1])
    print('detected feature dim :', actual_dim)
    if actual_dim != expected_dim:
        raise RuntimeError(f'Config expects input_dim={expected_dim}, but extracted features have dim={actual_dim}.')
PY


### 5. Generate CLIP Label Embeddings if Missing


In [ ]:
%%bash
set -e
cd /content/Deep_learning_project
PYTHONPATH=/content/Deep_learning_project python3.10 - <<'PY'
from pathlib import Path
import yaml
import torch

from src.clip_utils import save_class_embeddings_numpy

config_path = Path('configs/train_mlp_s3dis.yaml')
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
data_cfg = cfg['data']
clip_cfg = cfg.get('clip', {})
clip_emb_path = Path(data_cfg['label_embeddings_path'])

if clip_emb_path.exists():
    print('CLIP label embeddings already exist:', clip_emb_path)
else:
    label_texts = data_cfg.get('label_texts')
    if not label_texts:
        raise ValueError('`data.label_texts` is required to precompute label embeddings.')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    array = save_class_embeddings_numpy(
        output_path=clip_emb_path,
        class_names=label_texts,
        templates=clip_cfg.get('templates'),
        model_name=clip_cfg.get('model_name', 'ViT-B-32'),
        pretrained=clip_cfg.get('pretrained', 'openai'),
        device=device,
    )
    print(f'Saved CLIP label embeddings to {clip_emb_path} (shape: {array.shape})')
PY


### 6. Inspect Training Config


In [ ]:
!cat /content/Deep_learning_project/configs/train_mlp_s3dis.yaml


### 7. Train the Translation Head


In [ ]:
!PYTHONPATH=/content/Deep_learning_project python3.10 /content/Deep_learning_project/src/train.py --config /content/Deep_learning_project/configs/train_mlp_s3dis.yaml


### 8. Inspect Training Outputs


In [ ]:
%%bash
set -e
cd /content/Deep_learning_project
python3.10 - <<'PY'
from pathlib import Path
import json

ckpt_dir = Path('checkpoints/mlp_s3dis')
print('checkpoint dir exists:', ckpt_dir.exists())
for path in sorted(ckpt_dir.glob('*')):
    print(path.name)

history_path = ckpt_dir / 'history.json'
if history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))
    print('\nLast history rows:')
    for row in history[-5:]:
        print(row)
else:
    print('\nNo history.json found yet.')
PY
